In [1]:
import numpy as np
import pandas as pd
import pickle
from datetime import datetime

from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42

### Task 1: Data Loading

In [2]:
df = pd.read_csv("players.csv")
print("Shape:", df.shape)
print(df.head())

Shape: (50149, 26)
   player_id first_name     last_name                name  last_season  \
0         10   Miroslav         Klose      Miroslav Klose         2015   
1         26      Roman  Weidenfeller  Roman Weidenfeller         2017   
2         65    Dimitar      Berbatov    Dimitar Berbatov         2015   
3         77        NaN         Lúcio               Lúcio         2012   
4         80        Tom        Starke          Tom Starke         2017   

   current_club_id         player_code    country_of_birth city_of_birth  \
0              398      miroslav-klose              Poland         Opole   
1               16  roman-weidenfeller             Germany          Diez   
2             1091    dimitar-berbatov            Bulgaria   Blagoevgrad   
3              506               lucio              Brazil      Brasília   
4               27          tom-starke  East Germany (GDR)       Freital   

  country_of_citizenship  ...            agent_name  \
0                Germany

### Task 2: Data Preprocessing

In [ ]:
# Drop rows with missing target (market_value_in_eur)
df = df.dropna(subset=["market_value_in_eur"]).copy()
print(f"Dropped rows with missing target. Remaining rows: {len(df)}")
# Feature engineering — derive 'age' from date_of_birth 
df["date_of_birth"] = pd.to_datetime(df["date_of_birth"], errors="coerce")
reference_date = pd.Timestamp("2024-01-01")
df["age"] = (reference_date - df["date_of_birth"]).dt.days / 365.25
df["age"] = df["age"].fillna(df["age"].median())
print("Engineered 'age' feature from date_of_birth.")

# Feature engineering — years remaining on contract
df["contract_expiration_date"] = pd.to_datetime(df["contract_expiration_date"], errors="coerce")
df["years_to_contract_expiry"] = (df["contract_expiration_date"] - reference_date).dt.days / 365.25
df["years_to_contract_expiry"] = df["years_to_contract_expiry"].clip(lower=0)
df["years_to_contract_expiry"] = df["years_to_contract_expiry"].fillna(
    df["years_to_contract_expiry"].median()
)
print("Engineered 'years_to_contract_expiry' feature.")

# Handle missing values in categorical/numeric columns
df["foot"] = df["foot"].fillna("Unknown")
df["sub_position"] = df["sub_position"].fillna("Unknown")
df["height_in_cm"] = df["height_in_cm"].fillna(df["height_in_cm"].median())
print("Step 2.4 - Imputed missing values (foot, sub_position, height_in_cm).")

# Reduce high-cardinality categorical (country_of_citizenship)
# Keep top 20 countries, group the rest as 'Other' to avoid an explosion of
# one-hot columns and to reduce noise from rare categories.
top_countries = df["country_of_citizenship"].value_counts().nlargest(20).index
df["country_of_citizenship"] = df["country_of_citizenship"].where(
    df["country_of_citizenship"].isin(top_countries), "Other"
)
print("Reduced country_of_citizenship cardinality to top 20 + 'Other'.")

# Outlier handling — log-transform the skewed target
# market_value_in_eur is heavily right-skewed (a few superstar outliers).
# log1p compresses the scale and stabilizes variance for regression.
df["log_market_value"] = np.log1p(df["market_value_in_eur"])
print("Applied log1p transform to target to handle right-skew/outliers.")

# Drop irrelevant / leakage / identifier columns
drop_cols = [
    "player_id", "first_name", "last_name", "name", "player_code",
    "city_of_birth", "country_of_birth", "date_of_birth",
    "contract_expiration_date", "agent_name", "image_url", "url",
    "current_club_name", "current_club_id", "current_national_team_id",
    "last_season", "market_value_in_eur", "highest_market_value_in_eur",
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print("Dropped identifier/leakage/irrelevant columns.")
print("Remaining columns:", list(df.columns))

# Define feature groups
numeric_features = ["age", "height_in_cm", "years_to_contract_expiry", "international_caps", "international_goals"]
numeric_features = [c for c in numeric_features if c in df.columns]

categorical_features = ["position", "sub_position", "foot", "country_of_citizenship", "current_club_domestic_competition_id"]
categorical_features = [c for c in categorical_features if c in df.columns]

target_col = "log_market_value"

X = df[numeric_features + categorical_features]
y = df[target_col]

print(f"\nFinal feature set -> numeric: {numeric_features}, categorical: {categorical_features}")

Step 2.1 - Dropped rows with missing target. Remaining rows: 41528
Step 2.2 - Engineered 'age' feature from date_of_birth.
Step 2.3 - Engineered 'years_to_contract_expiry' feature.
Step 2.4 - Imputed missing values (foot, sub_position, height_in_cm).
Step 2.5 - Reduced country_of_citizenship cardinality to top 20 + 'Other'.
Step 2.6 - Applied log1p transform to target to handle right-skew/outliers.
Step 2.7 - Dropped identifier/leakage/irrelevant columns.
Remaining columns: ['country_of_citizenship', 'sub_position', 'position', 'foot', 'height_in_cm', 'international_caps', 'international_goals', 'current_club_domestic_competition_id', 'age', 'years_to_contract_expiry', 'log_market_value']

Final feature set -> numeric: ['age', 'height_in_cm', 'years_to_contract_expiry', 'international_caps', 'international_goals'], categorical: ['position', 'sub_position', 'foot', 'country_of_citizenship', 'current_club_domestic_competition_id']


### Task 3 Pipeline Creation

In [4]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
 
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
 
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

### Task 4: Primary Model Selection

In [5]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
])

### Task 5: Model Training

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

model.fit(X_train, y_train)
print("Baseline RandomForestRegressor trained.")

Train size: (33222, 10), Test size: (8306, 10)
Baseline RandomForestRegressor trained.


### Task 6: Cross-Validation

In [7]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="r2", n_jobs=-1)
print(f"5-Fold CV R^2 scores: {np.round(cv_scores, 4)}")
print(f"Mean R^2: {cv_scores.mean():.4f}  |  Std Dev: {cv_scores.std():.4f}")

5-Fold CV R^2 scores: [0.5723 0.558  0.5587 0.559  0.574 ]
Mean R^2: 0.5644  |  Std Dev: 0.0072


### Task 7: Hyperparameter Tuning

In [8]:
param_distributions = {
    "regressor__n_estimators": [100, 200, 300],
    "regressor__max_depth": [None, 10, 20, 30],
    "regressor__min_samples_split": [2, 5, 10],
    "regressor__min_samples_leaf": [1, 2, 4],
    "regressor__max_features": ["sqrt", "log2", None],
}
 
search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    n_iter=15,
    cv=3,
    scoring="r2",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train, y_train)
 
print("\nParameters tested (sample of results):")
results_df = pd.DataFrame(search.cv_results_)[
    ["params", "mean_test_score", "std_test_score", "rank_test_score"]
].sort_values("rank_test_score")
print(results_df.to_string(index=False))
 
print("\nBest parameters found:", search.best_params_)
print(f"Best CV R^2 score: {search.best_score_:.4f}")

Fitting 3 folds for each of 15 candidates, totalling 45 fits

Parameters tested (sample of results):
                                                                                                                                                                params  mean_test_score  std_test_score  rank_test_score
{'regressor__n_estimators': 100, 'regressor__min_samples_split': 2, 'regressor__min_samples_leaf': 2, 'regressor__max_features': 'sqrt', 'regressor__max_depth': None}         0.576756        0.006268                1
    {'regressor__n_estimators': 200, 'regressor__min_samples_split': 2, 'regressor__min_samples_leaf': 2, 'regressor__max_features': None, 'regressor__max_depth': 20}         0.568883        0.011216                2
    {'regressor__n_estimators': 100, 'regressor__min_samples_split': 5, 'regressor__min_samples_leaf': 1, 'regressor__max_features': None, 'regressor__max_depth': 20}         0.568697        0.011491                3
  {'regressor__n_estimators': 3

### Task 8: Best Model Selection

In [9]:
best_model = search.best_estimator_
print("Selected the estimator with the highest mean CV R^2 from RandomizedSearchCV:")
print(best_model.named_steps["regressor"])

Selected the estimator with the highest mean CV R^2 from RandomizedSearchCV:
RandomForestRegressor(max_features='sqrt', min_samples_leaf=2, n_jobs=-1,
                      random_state=42)


### Task 9: MODEL PERFORMANCE EVALUATION

In [10]:
y_pred_log = best_model.predict(X_test)

# Invert the log1p transform to get predictions back in EUR for reporting
y_test_eur = np.expm1(y_test)
y_pred_eur = np.expm1(y_pred_log)

mae_log = mean_absolute_error(y_test, y_pred_log)
rmse_log = np.sqrt(mean_squared_error(y_test, y_pred_log))
r2_log = r2_score(y_test, y_pred_log)

mae_eur = mean_absolute_error(y_test_eur, y_pred_eur)
rmse_eur = np.sqrt(mean_squared_error(y_test_eur, y_pred_eur))

print("Metrics on log-transformed target (model's native scale):")
print(f"  MAE  : {mae_log:.4f}")
print(f"  RMSE : {rmse_log:.4f}")
print(f"  R^2  : {r2_log:.4f}")

print("\nMetrics converted back to EUR (business-interpretable scale):")
print(f"  MAE  : €{mae_eur:,.0f}")
print(f"  RMSE : €{rmse_eur:,.0f}")
 
# Feature importances
ohe = best_model.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
cat_feature_names = list(ohe.get_feature_names_out(categorical_features))
all_feature_names = numeric_features + cat_feature_names
importances = best_model.named_steps["regressor"].feature_importances_
importance_df = pd.DataFrame({
    "feature": all_feature_names,
    "importance": importances
}).sort_values("importance", ascending=False).head(15)
print("\nTop 15 feature importances:")
print(importance_df.to_string(index=False))

Metrics on log-transformed target (model's native scale):
  MAE  : 0.7532
  RMSE : 0.9742
  R^2  : 0.5907

Metrics converted back to EUR (business-interpretable scale):
  MAE  : €1,110,036
  RMSE : €5,514,862

Top 15 feature importances:
                                  feature  importance
                 years_to_contract_expiry    0.284978
                       international_caps    0.125483
                                      age    0.123491
                      international_goals    0.057685
                             height_in_cm    0.048206
 current_club_domestic_competition_id_GB1    0.033610
            country_of_citizenship_Turkey    0.021517
                             foot_Unknown    0.020298
             country_of_citizenship_Other    0.017125
 current_club_domestic_competition_id_IT1    0.012296
  current_club_domestic_competition_id_L1    0.011931
 current_club_domestic_competition_id_TR1    0.011404
 current_club_domestic_competition_id_FR1    0.010321
curren

In [11]:
# ============================================================
# Save model (For Gradio)
# ============================================================
with open("player_rf_pipeline.pkl", "wb") as f:
    pickle.dump(best_model, f)
 
print("\n✅ Random Forest pipeline saved as player_rf_pipeline.pkl")


✅ Random Forest pipeline saved as player_rf_pipeline.pkl
